# Music Annotation Project 59R5
## Expanded Multi-Scale Retained-Region Challenger

**Runtime:** GPU recommended. Select a T4 GPU and run all.

59R5 is a development-only challenger. It consumes the Notebook 59R4 evidence and optional 59R4A adjudications, excludes `REGION_FRAGMENT` from supervised fitting, keeps low-support classes in the audit, and compares:

1. scalar observable baseline;
2. local 33×33 CNN;
3. multi-scale 33×33 / 97×97 / 257×257 CNN;
4. multi-scale CNN with observable local region merging.

Primary semantic classes:

`ANNOTATION / PRINT / MIXED / PAGE_SCAN_EDGE / UNCERTAIN`

The annotation-family head remains diagnostic only.

No model from this notebook has promotion authority. Page identity, raw coordinates, source-group identity, sampling-family names, 59R3 flags and clean/truth-derived inputs are prohibited predictors.


In [1]:
# @title
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy scikit-learn opencv-python-headless torch torchvision matplotlib pyarrow joblib


Mounted at /content/drive


In [2]:
# @title
import os
os.environ['CUBLAS_WORKSPACE_CONFIG']=':4096:8'
os.environ['PYTHONHASHSEED']='42'
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, random, shutil, zipfile, sys, platform, gc
import numpy as np
import pandas as pd
import cv2, torch, joblib
import matplotlib.pyplot as plt
from torch import nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import balanced_accuracy_score,f1_score,confusion_matrix,classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
R59R4_ROOT=ROOT/'notebook59R4_multiscale_region_semantics'
R59R4A_ROOT=ROOT/'region_semantic_adjudication_59R4A'
OUTPUT_ROOT=ROOT/'region_semantic_challenger_59R5'
SOURCE_59R4_RUN_OVERRIDE=None
SOURCE_59R4A_RUN_OVERRIDE=None
ADJUDICATION_CSV_OVERRIDE=None
SEED=42; EPOCHS=15; BATCH=32; LR=1e-3; WEIGHT_DECAY=1e-4; FOLDS=5
MIN_CLASS_REGIONS=10; MIN_CLASS_PAGES=5
MERGE_GAP=18; MERGE_DILATION=9
SCALES=[33,97,257]
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_59R5_expanded_multiscale'
OUT=OUTPUT_ROOT/RUN; INPUTS=OUT/'inputs'; TABLES=OUT/'tables'; MODELS=OUT/'models'; VISUAL=OUT/'visuals'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'
for p in [OUT,INPUTS,TABLES,MODELS,VISUAL,AUDIT,UPLOAD]:p.mkdir(parents=True,exist_ok=False)

def require(ok,msg):
    if not ok:raise RuntimeError(msg)
def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):h.update(b)
    return h.hexdigest()
def atomic_json(path,value):
    tmp=Path(str(path)+'.tmp');tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str));tmp.replace(path)
def reset_seed(seed=SEED):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.cuda.manual_seed_all(seed)
def latest_complete(root,required):
    hits=[]
    if Path(root).exists():
        for p in Path(root).rglob('*'):
            if p.is_dir() and all((p/x).exists() for x in required):hits.append(p)
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

def find_unique_below(root,name,label,required=True):
    hits=sorted({p.resolve() for p in Path(root).rglob(name) if p.is_file()})
    if not hits and not required:return None
    require(len(hits)==1,f'{label}: expected one {name}, found {len(hits)}: {hits[:20]}')
    return hits[0]

R4=Path(SOURCE_59R4_RUN_OVERRIDE) if SOURCE_59R4_RUN_OVERRIDE else latest_complete(R59R4_ROOT,['tables/out_of_fold_predictions.csv','tables/model_summary.csv','audit/notebook59R4_verdict.json'])
require(R4 is not None,'No complete 59R4 run found.')
R4A=Path(SOURCE_59R4A_RUN_OVERRIDE) if SOURCE_59R4A_RUN_OVERRIDE else latest_complete(R59R4A_ROOT,['tables/review_adjudications.csv','notebook59R4A_verdict.json'])
PRED_PATH=R4/'tables'/'out_of_fold_predictions.csv'
DECISIONS_PATH=find_unique_below(R4,'notebook66_completed_snapshot.csv','Notebook 66 decisions')
REGIONS_PATH=find_unique_below(R4,'retained_regions_clustered.csv','59R3 retained regions',required=False)
if REGIONS_PATH is None: REGIONS_PATH=find_unique_below(R4,'reviewed_regions_joined.csv','59R4 region dataset')
ADJ_PATH=Path(ADJUDICATION_CSV_OVERRIDE) if ADJUDICATION_CSV_OVERRIDE else ((R4A/'tables'/'review_adjudications.csv') if R4A is not None else None)
if ADJ_PATH is not None and not ADJ_PATH.exists():ADJ_PATH=None
print('59R4:',R4);print('59R4A:',R4A);print('Decisions:',DECISIONS_PATH);print('Regions:',REGIONS_PATH);print('Adjudications:',ADJ_PATH);print('Device:',DEVICE);print('Output:',OUT)


59R4: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4
59R4A: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_093214_59R4A_targeted_adjudication
Decisions: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4/inputs/notebook66_completed_snapshot.csv
Regions: /content/drive/MyDrive/MusicAnnotationProject/notebook59R4_multiscale_region_semantics/20261114_165044_notebook59R4/inputs/reviewed_regions_joined.csv
Adjudications: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_adjudication_59R4A/20261115_093214_59R4A_targeted_adjudication/tables/review_adjudications.csv
Device: cuda
Output: /content/drive/MyDrive/MusicAnnotationProject/region_semantic_challenger_59R5/20261115_110552_59R5_expanded_multiscale


In [3]:
# @title
pred=pd.read_csv(PRED_PATH,dtype=str).fillna('')
dec=pd.read_csv(DECISIONS_PATH,dtype=str).fillna('')
regions=pd.read_csv(REGIONS_PATH,low_memory=False)
for f,n in [(dec,'review decisions'),(pred,'59R4 predictions'),(regions,'region table')]:require('region_id' in f.columns,f'{n} lacks region_id')
completed=dec.merge(regions,on='region_id',how='left',validate='one_to_one',suffixes=('','_region')).merge(pred,on='region_id',how='left',validate='one_to_one',suffixes=('','_59R4'))
completed['original_semantic_label']=completed.semantic_label.astype(str)
completed['original_annotation_family']=completed.annotation_family.astype(str)
if ADJ_PATH is not None:
    adj=pd.read_csv(ADJ_PATH,dtype=str).fillna('')
    require({'region_id','adjudicated_semantic_label'}.issubset(adj.columns),'Adjudication schema invalid')
    require(not adj.region_id.duplicated().any(),'Duplicate adjudication region_id')
    completed=completed.merge(adj,on='region_id',how='left',validate='one_to_one')
    use=completed.adjudicated_semantic_label.astype(str).str.strip().ne('')
    completed.loc[use,'semantic_label']=completed.loc[use,'adjudicated_semantic_label']
    if 'adjudicated_annotation_family' in completed:
        uf=completed.adjudicated_annotation_family.astype(str).str.strip().ne('')
        completed.loc[uf,'annotation_family']=completed.loc[uf,'adjudicated_annotation_family']
completed['semantic_label']=completed.semantic_label.astype(str).str.upper().str.strip()
completed['annotation_family']=completed.annotation_family.astype(str).str.upper().str.strip()
completed['fit_exclusion_reason']=''
completed.loc[completed.semantic_label.eq('REGION_FRAGMENT'),'fit_exclusion_reason']='REGION_FRAGMENT'
completed.loc[completed.semantic_label.eq('MASK_ERROR'),'fit_exclusion_reason']='MASK_ERROR'

# Support audit and minimum-support gate.
support=completed.groupby('semantic_label',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'))
support['eligible_for_supervised_fit']=(support.regions>=MIN_CLASS_REGIONS)&(support.pages>=MIN_CLASS_PAGES)&~support.semantic_label.isin(['REGION_FRAGMENT','MASK_ERROR'])
support.to_csv(TABLES/'semantic_class_support.csv',index=False)
eligible_classes=support.loc[support.eligible_for_supervised_fit,'semantic_label'].tolist()
completed.loc[~completed.semantic_label.isin(eligible_classes)&completed.fit_exclusion_reason.eq(''),'fit_exclusion_reason']='INSUFFICIENT_CLASS_SUPPORT'
fit=completed[completed.fit_exclusion_reason.eq('')].copy()
require(fit.semantic_label.nunique()>=3,f'Fewer than three supported semantic classes: {eligible_classes}')
require(fit.page.nunique()>=10,'Too few pages for grouped OOF')
completed.to_csv(INPUTS/'effective_review_evidence.csv',index=False)
print(support.to_string(index=False));print('Fit rows:',len(fit),'pages:',fit.page.nunique(),'classes:',eligible_classes)


semantic_label  regions  pages  eligible_for_supervised_fit
    ANNOTATION       32     32                         True
         MIXED        5      5                        False
           NAN      140     92                         True
PAGE_SCAN_EDGE       20     19                         True
         PRINT        2      2                        False
     UNCERTAIN        1      1                        False
Fit rows: 192 pages: 114 classes: ['ANNOTATION', 'NAN', 'PAGE_SCAN_EDGE']


In [4]:
# @title
# Predictor firewall and path resolution.
PROHIBITED_EXACT={'page','region_id','source_group','sampling_family','semantic_label','annotation_family','original_semantic_label','original_annotation_family','review_order','review_id','reviewed_utc','cluster_id','x','y','x0','y0','x1','y1','x_fraction','y_fraction'}
PROHIBITED_PATTERNS=['flag_','_flag_','human','truth','clean','outcome','failure','pred_','prob_','sampling_family','source_group','page_','adjudicat','review']

def first_col(frame,names):return next((c for c in names if c in frame.columns),None)
def numeric(names,default=np.nan):
    c=first_col(fit,names);return pd.to_numeric(fit[c],errors='coerce') if c else pd.Series(default,index=fit.index,dtype=float)
fit['_x0']=numeric(['x0','bbox_x0','x','left']);fit['_y0']=numeric(['y0','bbox_y0','y','top']);fit['_x1']=numeric(['x1','bbox_x1','right']);fit['_y1']=numeric(['y1','bbox_y1','bottom'])
if fit['_x1'].isna().all():fit['_x1']=fit['_x0']+numeric(['bbox_w','w','width'])
if fit['_y1'].isna().all():fit['_y1']=fit['_y0']+numeric(['bbox_h','h','height'])
require(fit[['_x0','_y0','_x1','_y1']].notna().all().all(),'Region geometry is incomplete')

# ============================================================
# Resolve authoritative observed pages
#
# 59R5 must not assume that observed pages were physically
# copied into the 59R4 run.
#
# Resolution priority:
#
#   1. explicit durable image path already present in row;
#   2. Notebook 07 review-audit aligned_output;
#   3. ROOT/aligned exact page stem;
#   4. ROOT/source/annotated_only exact page stem;
#
# Do not search 59R4 visual/contact-sheet outputs as if they
# were authoritative page inputs.
# ============================================================

SUPPORTED_IMAGES = {
    '.png',
    '.jpg',
    '.jpeg',
    '.tif',
    '.tiff',
    '.bmp',
}


# ------------------------------------------------------------
# 1. Helpers
# ------------------------------------------------------------

def image_map(
    folder,
):

    folder = Path(
        folder
    )

    if not folder.exists():

        return {}


    result = {}


    for path in folder.iterdir():

        if (
            path.is_file()
            and path.suffix.lower()
            in SUPPORTED_IMAGES
        ):

            stem = path.stem


            require(
                stem not in result,
                (
                    f'Duplicate page stem {stem!r} '
                    f'in {folder}.'
                ),
            )


            result[
                stem
            ] = path


    return result


def existing_image_path(
    value,
):

    if value is None:

        return None


    text = str(
        value
    ).strip()


    if (
        not text
        or text.lower()
        in {
            'nan',
            'none',
        }
    ):

        return None


    path = Path(
        text
    )


    if (
        path.is_file()
        and path.suffix.lower()
        in SUPPORTED_IMAGES
    ):

        return path


    # Relative to project root.

    candidate = (
        ROOT
        / path
    )


    if (
        candidate.is_file()
        and candidate.suffix.lower()
        in SUPPORTED_IMAGES
    ):

        return candidate


    return None


# ------------------------------------------------------------
# 2. Direct authoritative page maps
# ------------------------------------------------------------

ALIGNED_MAP = image_map(
    ROOT
    / 'aligned'
)


ANNOTATED_ONLY_MAP = image_map(
    ROOT
    / 'source'
    / 'annotated_only'
)


print(
    'ROOT/aligned image pages:',
    len(
        ALIGNED_MAP
    )
)

print(
    'ROOT/source/annotated_only image pages:',
    len(
        ANNOTATED_ONLY_MAP
    )
)


# ------------------------------------------------------------
# 3. Notebook 07 review audits
#
# Notebook 07 reviewed pages record the aligned page used by
# the corrected-mask workflow as aligned_output.
# ------------------------------------------------------------

REVIEW_AUDIT_DIR = (
    ROOT
    / 'human_in_the_middle'
    / 'audit'
)


review_audits = []


if REVIEW_AUDIT_DIR.exists():

    for audit_path in sorted(
        REVIEW_AUDIT_DIR.glob(
            '*.json'
        )
    ):

        try:

            record = json.loads(
                audit_path.read_text()
            )

        except Exception as error:

            raise RuntimeError(
                'Unreadable Notebook 07 review audit: '
                f'{audit_path}: {error}'
            )


        if not isinstance(
            record,
            dict,
        ):

            raise RuntimeError(
                'Notebook 07 review audit root is not '
                f'an object: {audit_path}'
            )


        record[
            '_audit_path'
        ] = str(
            audit_path
        )


        review_audits.append(
            record
        )


print(
    'Notebook 07 review audit records:',
    len(
        review_audits
    )
)


# ------------------------------------------------------------
# 4. Build audit lookup by safe page identities
# ------------------------------------------------------------

AUDIT_BY_PAGE = {}


def add_page_identity(
    raw_identity,
    record,
):

    if raw_identity is None:

        return


    text = str(
        raw_identity
    ).strip()


    if not text:

        return


    try:

        page = Path(
            text
        ).stem

    except Exception:

        page = text


    if not page:

        return


    AUDIT_BY_PAGE.setdefault(
        page,
        []
    )


    if record not in AUDIT_BY_PAGE[
        page
    ]:

        AUDIT_BY_PAGE[
            page
        ].append(
            record
        )


for record in review_audits:

    # Direct page identifiers.

    for key in [
        'page',
        'page_id',
        'source_stem',
        'stem',
        'name',
    ]:

        if record.get(
            key
        ):

            add_page_identity(
                record[
                    key
                ],
                record,
            )


    # File paths can recover the page stem as well.

    for key in [
        'source',
        'source_file',
        'aligned_output',
        'corrected_mask_output',
    ]:

        if record.get(
            key
        ):

            add_page_identity(
                record[
                    key
                ],
                record,
            )


# ------------------------------------------------------------
# 5. Resolve Notebook 07 aligned page
# ------------------------------------------------------------

def resolve_notebook07_observed(
    page,
):

    records = (
        AUDIT_BY_PAGE.get(
            str(
                page
            ),
            []
        )
    )


    paths = []


    for record in records:

        path = existing_image_path(
            record.get(
                'aligned_output'
            )
        )


        if path is not None:

            paths.append(
                path.resolve()
            )


    paths = sorted(
        set(
            paths
        )
    )


    # If multiple historical audit records resolve for the
    # same page, require byte identity rather than picking one
    # by modification time.

    if len(
        paths
    ) > 1:

        hashes = {
            sha256_file(
                path
            )
            for path
            in paths
        }


        require(
            len(
                hashes
            )
            == 1,
            (
                f'{page}: multiple non-identical Notebook 07 '
                f'aligned_output pages resolve: {paths}'
            ),
        )


    return (
        paths[
            0
        ]
        if paths
        else None
    )


# ------------------------------------------------------------
# 6. Explicit image-path columns potentially carried through
# 59R3 / Notebook 66 / 59R4.
# ------------------------------------------------------------

ROW_IMAGE_COLUMNS = [
    'observed_path',
    'observed_image_path',
    'page_path',
    'page_image_path',
    'aligned_path',
    'aligned_output',
    'source_path',
    'source_file',
    'image_path',
]


# ------------------------------------------------------------
# 7. Main authoritative resolver
# ------------------------------------------------------------

def resolve_page(
    row,
):

    page = str(
        row[
            'page'
        ]
    ).strip()


    # A. Exact path already persisted in the joined evidence.

    for column in ROW_IMAGE_COLUMNS:

        if column in row.index:

            path = existing_image_path(
                row[
                    column
                ]
            )


            if path is not None:

                return (
                    path,
                    f'row:{column}',
                )


    # B. Notebook 07 reviewed aligned output.
    #
    # Prefer this over generic aligned copies because this
    # records the actual page associated with human review.

    path = resolve_notebook07_observed(
        page
    )


    if path is not None:

        return (
            path,
            'Notebook07:aligned_output',
        )


    # C. Project-level aligned image.

    if page in ALIGNED_MAP:

        return (
            ALIGNED_MAP[
                page
            ],
            'ROOT/aligned',
        )


    # D. Original annotated-only input.
    #
    # This is the final legitimate fallback. Do not substitute
    # contact sheets, threshold ladders or 59R4 crops for the
    # observed page.

    if page in ANNOTATED_ONLY_MAP:

        return (
            ANNOTATED_ONLY_MAP[
                page
            ],
            'ROOT/source/annotated_only',
        )


    return (
        None,
        None,
    )


# ------------------------------------------------------------
# 8. Resolve each unique page only once
# ------------------------------------------------------------

page_resolution_rows = []

PAGE_PATHS = {}


for page in sorted(
    fit[
        'page'
    ]
    .astype(str)
    .unique()
):

    representative = (
        fit[
            fit[
                'page'
            ].astype(str)
            == page
        ]
        .iloc[
            0
        ]
    )


    path, resolution_source = resolve_page(
        representative
    )


    PAGE_PATHS[
        page
    ] = path


    page_resolution_rows.append({
        'page':
            page,

        'observed_path':
            (
                str(
                    path
                )
                if path is not None
                else ''
            ),

        'resolution_source':
            (
                resolution_source
                or ''
            ),

        'resolved':
            bool(
                path is not None
            ),
    })


page_resolution = pd.DataFrame(
    page_resolution_rows
)


page_resolution.to_csv(
    AUDIT
    / 'observed_page_resolution.csv',
    index=False,
)


missing_pages = (  page_resolution[
        ~page_resolution[
            'resolved'
        ]
    ]
    .copy()
)


if len(
    missing_pages
):

    print(
        '\nUnresolved observed pages:'
    )

    print(
        missing_pages[
            'page'
        ].to_list()
    )


require(
    len(
        missing_pages
    )
    == 0,
    (
        f'{len(missing_pages)} unique observed pages '
        'could not be resolved. '
        'See audit/observed_page_resolution.csv.'
    ),
)


# ------------------------------------------------------------
# 9. Attach resolved page path to every fitted region
# ------------------------------------------------------------

fit[
    '_observed_path'
] = (
    fit[
        'page'
    ]
    .astype(str)
    .map(
        PAGE_PATHS
    )
)


require(
    fit[
        '_observed_path'
    ].notna().all(),
    (
        'One or more fitted rows lack an observed page '
        'after page-level resolution.'
    ),
)


# ------------------------------------------------------------
# 10. Verify readability and geometry against each page
# ------------------------------------------------------------

page_identity_rows = []


for page, group in fit.groupby(
    'page',
    sort=True,
):

    page = str(
        page
    )


    path = Path(
        PAGE_PATHS[
            page
        ]
    )


    observed = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    require(
        observed is not None,
        (
            f'{page}: resolved observed page is unreadable: '
            f'{path}'
        ),
    )


    height, width = (
        observed.shape
    )


    x0 = group[
        '_x0'
    ].to_numpy(
        dtype=float
    )

    y0 = group[
        '_y0'
    ].to_numpy(
        dtype=float
    )

    x1 = group[
        '_x1'
    ].to_numpy(
        dtype=float
    )

    y1 = group[
        '_y1'
    ].to_numpy(
        dtype=float
    )


    require(
        np.isfinite(
            np.column_stack(
                [
                    x0,
                    y0,
                    x1,
                    y1,
                ]
            )
        ).all(),
        (
            f'{page}: non-finite region geometry.'
        ),
    )


    invalid = (
        (
            x0 < 0
        )
        | (
            y0 < 0
        )
        | (
            x1 > width
        )
        | (
            y1 > height
        )
        | (
            x1 <= x0
        )
        | (
            y1 <= y0
        )
    )


    require(
        not invalid.any(),
        (
            f'{page}: '
            f'{int(invalid.sum())} candidate regions '
            'do not fit inside the resolved observed page '
            f'of shape {observed.shape}.'
        ),
    )


    page_identity_rows.append({
        'page':
            page,

        'observed_path':
            str(
                path
            ),

        'height':
            int(
                height
            ),

        'width':
            int(
                width
            ),

        'region_rows':
            int(
                len(
                    group
                )
            ),

        'sha256':
            sha256_file(
                path
            ),
    })


pd.DataFrame(
    page_identity_rows
).to_csv(
    AUDIT
    / 'resolved_observed_page_identity.csv',
    index=False,
)


print(
    '\nObserved-page resolution sources:'
)


print(
    page_resolution[
        'resolution_source'
    ].value_counts()
)


print(
    '\nResolved',
    len(
        PAGE_PATHS
    ),
    'unique observed pages for',
    len(
        fit
    ),
    'fitted regions.'
)

# Numeric scalar baseline excludes all identities, coordinates, automatic flags, labels and post-outcome fields.
scalar=[]
for c in fit.select_dtypes(include=[np.number,bool]).columns:
    lc=c.lower()
    if c in PROHIBITED_EXACT or c.startswith('_'):continue
    if any(p in lc for p in PROHIBITED_PATTERNS):continue
    scalar.append(c)
require(len(scalar)>0,'No scalar predictors remain after firewall')
firewall={'eligible_scalar_predictors':scalar,'prohibited_exact':sorted(PROHIBITED_EXACT),'prohibited_patterns':PROHIBITED_PATTERNS,'raw_coordinates_excluded':True,'page_identity_excluded':True,'source_group_excluded':True,'59R3_flags_excluded':True,'truth_derived_inputs_excluded':True}
atomic_json(AUDIT/'predictor_firewall.json',firewall)
print('Scalar predictors:',scalar)


ROOT/aligned image pages: 260
ROOT/source/annotated_only image pages: 173
Notebook 07 review audit records: 173

Observed-page resolution sources:
resolution_source
Notebook07:aligned_output    114
Name: count, dtype: int64

Resolved 114 unique observed pages for 192 fitted regions.
Scalar predictors: ['source_region_index_region', 'pixels_region', 'x_region', 'y_region', 'w_region', 'h_region', 'cx_region', 'cy_region', 'x_fraction_region', 'y_fraction_region', 'w_fraction_region', 'h_fraction_region', 'bbox_fill_region', 'aspect_ratio_region', 'major_axis_proxy_region', 'minor_axis_proxy_region', 'orientation_deg_region', 'linearity_region', 'skeleton_pixels_region', 'skeleton_to_pixels_region', 'endpoints_region', 'junctions_region', 'observed_mean_region', 'observed_median_region', 'observed_q25_region', 'observed_q75_region', 'observed_std_region', 'p_remove_mean_region', 'p_remove_median_region', 'p_remove_min_region', 'p_remove_max_region', 'p_remove_std_region', 'context_dark_d

In [5]:
# @title


In [ ]:
# @title
# ============================================================
# 59R5 PAGE-CACHED TENSOR BUILDER, EQUIVALENCE GATE
# AND PAGE-GROUPED OOF TRAINING
#
# REPLACES:
#   1. the old observable-image/tensor-builder cell; and
#   2. the old CNN OOF training cell.
#
# Assumes these already exist:
#
#   fit
#   classes
#   class_to_id, or enough information to reconstruct it
#   splits, or enough information to reconstruct them
#   scalar
#   MODELS
#   TABLES
#   AUDIT
#   DEVICE
#   SEED
#   EPOCHS
#   BATCH
#   LR
#   WEIGHT_DECAY
#   FOLDS
#   SCALES
#   MERGE_GAP
#   MERGE_DILATION
#   require()
#   reset_seed()
#
# Required fit columns:
#
#   region_id
#   page
#   semantic_label
#   _observed_path
#   _x0
#   _y0
#   _x1
#   _y1
#
# ============================================================

import gc
import time
import hashlib

from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
)
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import joblib


# ------------------------------------------------------------
# Configuration specific to the cached builder
# ------------------------------------------------------------

CACHE_EQUIVALENCE_SAMPLE_SIZE = min(
    24,
    len(
        fit
    ),
)

CACHE_EQUIVALENCE_ATOL = 0.0
CACHE_EQUIVALENCE_RTOL = 0.0

CACHE_PAD = (
    max(
        SCALES
    )
    // 2
    + 2
)

CACHE_CHANNELS = [
    'observed',
    'normalised',
    'region_mask',
    'region_distance',
    'edge',
    'valid_image',
]

CACHE_CHANNEL_COUNT = len(
    CACHE_CHANNELS
)

PIN_MEMORY = (
    DEVICE
    == 'cuda'
)

NUM_WORKERS = 0

CHECKPOINT_EVERY_FOLD = True


# ------------------------------------------------------------
# Validate and normalise the fitted evidence table
# ------------------------------------------------------------

required_tensor_columns = {
    'region_id',
    'page',
    'semantic_label',
    '_observed_path',
    '_x0',
    '_y0',
    '_x1',
    '_y1',
}


missing_tensor_columns = (
    required_tensor_columns
    - set(
        fit.columns
    )
)


require(
    not missing_tensor_columns,
    (
        'The page-cached tensor builder is missing columns: '
        f'{sorted(missing_tensor_columns)}'
    ),
)


fit = fit.reset_index(
    drop=True
).copy()


fit[
    'page'
] = (
    fit[
        'page'
    ]
    .astype(str)
)


fit[
    'region_id'
] = (
    fit[
        'region_id'
    ]
    .astype(str)
)


for column in [
    '_x0',
    '_y0',
    '_x1',
    '_y1',
]:

    fit[
        column
    ] = pd.to_numeric(
        fit[
            column
        ],
        errors='coerce',
    )


require(
    fit[
        [
            '_x0',
            '_y0',
            '_x1',
            '_y1',
        ]
    ].notna().all().all(),
    (
        'One or more retained regions has incomplete '
        'bounding-box geometry.'
    ),
)


# ------------------------------------------------------------
# Semantic class and fold setup
# ------------------------------------------------------------

classes = sorted(
    fit[
        'semantic_label'
    ]
    .astype(str)
    .unique()
    .tolist()
)


class_to_id = {
    class_name:
        class_id

    for class_id, class_name
    in enumerate(
        classes
    )
}


fit[
    '_class_id'
] = (
    fit[
        'semantic_label'
    ]
    .map(
        class_to_id
    )
    .astype(int)
)


groups = (
    fit[
        'page'
    ]
    .astype(str)
    .to_numpy()
)


number_of_folds = min(
    FOLDS,
    len(
        np.unique(
            groups
        )
    ),
)


require(
    number_of_folds
    >= 2,
    (
        'At least two page groups are required for OOF '
        'training.'
    ),
)


splitter = GroupKFold(
    n_splits=number_of_folds
)


splits = list(
    splitter.split(
        fit,
        fit[
            '_class_id'
        ],
        groups,
    )
)


fold_rows = []


for fold_number, (
    train_indices,
    test_indices,
) in enumerate(
    splits,
    start=1,
):

    train_pages = set(
        groups[
            train_indices
        ]
    )

    test_pages = set(
        groups[
            test_indices
        ]
    )


    overlap = (
        train_pages
        & test_pages
    )


    require(
        not overlap,
        (
            f'Fold {fold_number}: page leakage detected: '
            f'{sorted(overlap)}'
        ),
    )


    for row_index in test_indices:

        fold_rows.append({
            'region_id':
                fit.iloc[
                    row_index
                ][
                    'region_id'
                ],

            'page':
                fit.iloc[
                    row_index
                ][
                    'page'
                ],

            'fold':
                fold_number,
        })


pd.DataFrame(
    fold_rows
).to_csv(
    TABLES
    / 'fold_map.csv',
    index=False,
)


print(
    'Semantic classes:',
    classes,
)

print(
    'OOF folds:',
    len(
        splits
    ),
)


# ============================================================
# PAGE CACHE
# ============================================================

PAGE_CACHE = {}
PAGE_CACHE_AUDIT = []


def array_sha256(
    array,
):

    contiguous = np.ascontiguousarray(
        array
    )


    digest = hashlib.sha256()

    digest.update(
        str(
            contiguous.dtype
        ).encode()
    )

    digest.update(
        str(
            contiguous.shape
        ).encode()
    )

    digest.update(
        contiguous.tobytes()
    )


    return digest.hexdigest()


def read_observed_page(
    path,
):

    path = Path(
        path
    )


    require(
        path.is_file(),
        (
            'Observed page does not exist: '
            f'{path}'
        ),
    )


    image = cv2.imread(
        str(
            path
        ),
        cv2.IMREAD_GRAYSCALE,
    )


    require(
        image is not None,
        (
            'Observed page is unreadable: '
            f'{path}'
        ),
    )


    require(
        image.ndim
        == 2,
        (
            'Observed page is not a greyscale image: '
            f'{path}'
        ),
    )


    return image


def build_page_observable_arrays(
    observed,
):

    # Background estimate used by the original 59R5 builder.

    background = cv2.GaussianBlur(
        observed,
        (
            0,
            0,
        ),
        45,
    )


    normalised = cv2.divide(
        observed,
        np.maximum(
            background,
            1,
        ),
        scale=255,
    )


    edge = (
        cv2.Canny(
            observed,
            70,
            170,
        )
        .astype(
            np.float32
        )
        / 255.0
    )


    valid_image = np.ones(
        observed.shape,
        dtype=np.float32,
    )


    return {
        'observed':
            (
                observed.astype(
                    np.float32
                )
                / 255.0
            ),

        'normalised':
            (
                normalised.astype(
                    np.float32
                )
                / 255.0
            ),

        'edge':
            edge,

        'valid_image':
            valid_image,
    }


def validate_page_region_geometry(
    page,
    frame,
    shape,
):

    height, width = shape


    values = (
        frame[
            [
                '_x0',
                '_y0',
                '_x1',
                '_y1',
            ]
        ]
        .to_numpy(
            dtype=np.float64
        )
    )


    require(
        np.isfinite(
            values
        ).all(),
        (
            f'{page}: region geometry contains '
            'non-finite values.'
        ),
    )


    x0 = values[
        :,
        0
    ]

    y0 = values[
        :,
        1
    ]

    x1 = values[
        :,
        2
    ]

    y1 = values[
        :,
        3
    ]


    invalid = (
        (
            x0
            < 0
        )
        | (
            y0
            < 0
        )
        | (
            x1
            > width
        )
        | (
            y1
            > height
        )
        | (
            x1
            <= x0
        )
        | (
            y1
            <= y0
        )
    )


    require(
        not invalid.any(),
        (
            f'{page}: {int(invalid.sum())} retained-region '
            'boxes do not fit inside observed page shape '
            f'{shape}.'
        ),
    )


def single_region_mask(
    row,
    page_shape,
):

    mask = np.zeros(
        page_shape,
        dtype=np.uint8,
    )


    x0 = int(
        round(
            row[
                '_x0'
            ]
        )
    )

    y0 = int(
        round(
            row[
                '_y0'
            ]
        )
    )

    x1 = int(
        round(
            row[
                '_x1'
            ]
        )
    )

    y1 = int(
        round(
            row[
                '_y1'
            ]
        )
    )


    mask[
        y0:y1,
        x0:x1,
    ] = 1


    return mask


def boxes_are_merge_neighbours(
    first,
    second,
):

    horizontal_gap = max(
        0.0,
        max(
            float(
                first[
                    '_x0'
                ]
            ),
            float(
                second[
                    '_x0'
                ]
            ),
        )
        - min(
            float(
                first[
                    '_x1'
                ]
            ),
            float(
                second[
                    '_x1'
                ]
            ),
        ),
    )


    vertical_gap = max(
        0.0,
        max(
            float(
                first[
                    '_y0'
                ]
            ),
            float(
                second[
                    '_y0'
                ]
            ),
        )
        - min(
            float(
                first[
                    '_y1'
                ]
            ),
            float(
                second[
                    '_y1'
                ]
            ),
        ),
    )


    return (
        horizontal_gap
        <= MERGE_GAP
        and vertical_gap
        <= MERGE_GAP
    )


def merged_region_mask(
    row,
    page_frame,
    page_shape,
):

    mask = np.zeros(
        page_shape,
        dtype=np.uint8,
    )


    for _,candidate in page_frame.iterrows():

        if not boxes_are_merge_neighbours(
            row,
            candidate,
        ):

            continue


        x0 = int(
            round(
                candidate[
                    '_x0'
                ]
            )
        )

        y0 = int(
            round(
                candidate[
                    '_y0'
                ]
            )
        )

        x1 = int(
            round(
                candidate[
                    '_x1'
                ]
            )
        )

        y1 = int(
            round(
                candidate[
                    '_y1'
                ]
            )
        )


        mask[
            y0:y1,
            x0:x1,
        ] = 1


    if MERGE_DILATION > 0:

        kernel = np.ones(
            (
                MERGE_DILATION,
                MERGE_DILATION,
            ),
            dtype=np.uint8,
        )


        mask = cv2.dilate(
            mask,
            kernel,
        )


    return (
        mask
        > 0
    ).astype(
        np.uint8
    )


def distance_channel(
    mask,
):

    distance = cv2.distanceTransform(
        mask.astype(
            np.uint8
        ),
        cv2.DIST_L2,
        5,
    ).astype(
        np.float32
    )


    maximum = float(
        distance.max()
    )


    if maximum > 0:

        distance /= maximum


    return distance


def pad_channel(
    array,
    pad_mode,
):

    if pad_mode == 'constant':

        return np.pad(
            array,
            CACHE_PAD,
            mode='constant',
            constant_values=0,
        )


    return np.pad(
        array,
        CACHE_PAD,
        mode=pad_mode,
    )


def build_page_cache():

    started = time.time()


    for page, page_frame in fit.groupby(
        'page',
        sort=True,
    ):

        page_started = time.time()


        paths = {
            Path(
                path
            ).resolve()

            for path
            in page_frame[
                '_observed_path'
            ]
        }


        require(
            len(
                paths
            )
            == 1,
            (
                f'{page}: rows resolve to multiple observed '
                f'page paths: {sorted(paths)}'
            ),
        )


        observed_path = next(
            iter(
                paths
            )
        )


        observed = read_observed_page(
            observed_path
        )


        validate_page_region_geometry(
            page,
            page_frame,
            observed.shape,
        )


        observable_arrays = (
            build_page_observable_arrays(
                observed
            )
        )


        static_padded = {

            'observed':
                pad_channel(
                    observable_arrays[
                        'observed'
                    ],
                    'reflect',
                ),

            'normalised':
                pad_channel(
                    observable_arrays[
                        'normalised'
                    ],
                    'reflect',
                ),

            'edge':
                pad_channel(
                    observable_arrays[
                        'edge'
                    ],
                    'reflect',
                ),

            # Outside the real image must remain distinguishable
            # from actual page pixels.

            'valid_image':
                pad_channel(
                    observable_arrays[
                        'valid_image'
                    ],
                    'constant',
                ),
        }


        region_cache = {}


        for row_index, row in page_frame.iterrows():

            core_mask = single_region_mask(
                row,
                observed.shape,
            )


            merged_mask = merged_region_mask(
                row,
                page_frame,
                observed.shape,
            )


            core_distance = distance_channel(
                core_mask
            )


            merged_distance = distance_channel(
                merged_mask
            )


            region_cache[
                int(
                    row_index
                )
            ] = {

                'core_mask':
                    pad_channel(
                        core_mask.astype(
                            np.float32
                        ),
                        'constant',
                    ),

                'core_distance':
                    pad_channel(
                        core_distance,
                        'constant',
                    ),

                'merged_mask':
                    pad_channel(
                        merged_mask.astype(
                            np.float32
                        ),
                        'constant',
                    ),

                'merged_distance':
                    pad_channel(
                        merged_distance,
                        'constant',
                    ),
            }


        PAGE_CACHE[
            str(
                page
            )
        ] = {
            'observed_path':
                observed_path,

            'shape':
                observed.shape,

            'static':
                static_padded,

            'regions':
                region_cache,
        }


        PAGE_CACHE_AUDIT.append({
            'page':
                str(
                    page
                ),

            'observed_path':
                str(
                    observed_path
                ),

            'height':
                int(
                    observed.shape[
                        0
                    ]
                ),

            'width':
                int(
                    observed.shape[
                        1
                    ]
                ),

            'regions':
                int(
                    len(
                        page_frame
                    )
                ),

            'observed_sha256':
                array_sha256(
                    observed
                ),

            'cache_build_seconds':
                float(
                    time.time()
                    - page_started
                ),
        })


        print(
            f'Cached {page}: '
            f'{len(page_frame)} regions, '
            f'shape={observed.shape}'
        )


    audit_frame = pd.DataFrame(
        PAGE_CACHE_AUDIT
    )


    audit_frame.to_csv(
        AUDIT
        / 'page_tensor_cache_audit.csv',
        index=False,
    )


    print(
        '\nPage cache complete:',
        len(
            PAGE_CACHE
        ),
        'pages in',
        round(
            time.time()
            - started,
            2,
        ),
        'seconds.',
    )


build_page_cache()


# ============================================================
# PATCH EXTRACTION
# ============================================================

def region_centre(
    row,
):

    centre_y = int(
        round(
            (
                float(
                    row[
                        '_y0'
                    ]
                )
                + float(
                    row[
                        '_y1'
                    ]
                )
            )
            / 2.0
        )
    )


    centre_x = int(
        round(
            (
                float(
                    row[
                        '_x0'
                    ]
                )
                + float(
                    row[
                        '_x1'
                    ]
                )
            )
            / 2.0
        )
    )


    return (
        centre_y,
        centre_x,
    )


def cached_crop(
    padded_array,
    centre_y,
    centre_x,
    size,
):

    half = (
        size
        // 2
    )


    padded_y = (
        centre_y
        + CACHE_PAD
    )

    padded_x = (
        centre_x
        + CACHE_PAD
    )


    crop = padded_array[
        padded_y
        - half:
        padded_y
        + half
        + 1,

        padded_x
        - half:
        padded_x
        + half
        + 1,
    ]


    require(
        crop.shape
        == (
            size,
            size,
        ),
        (
            'Cached crop has wrong shape: '
            f'expected {(size, size)}, found {crop.shape}.'
        ),
    )


    return crop


def cached_channels(
    row_index,
    size,
    merged=False,
):

    row = fit.iloc[
        int(
            row_index
        )
    ]


    page = str(
        row[
            'page'
        ]
    )


    cache = PAGE_CACHE[
        page
    ]


    centre_y, centre_x = region_centre(
        row
    )


    region = cache[
        'regions'
    ][
        int(
            row_index
        )
    ]


    mask_key = (
        'merged_mask'
        if merged
        else 'core_mask'
    )


    distance_key = (
        'merged_distance'
        if merged
        else 'core_distance'
    )


    arrays = [
        cache[
            'static'
        ][
            'observed'
        ],

        cache[
            'static'
        ][
            'normalised'
        ],

        region[
            mask_key
        ],

        region[
            distance_key
        ],

        cache[
            'static'
        ][
            'edge'
        ],

        cache[
            'static'
        ][
            'valid_image'
        ],
    ]


    tensor = np.stack(
        [
            cached_crop(
                array,
                centre_y,
                centre_x,
                size,
            )

            for array
            in arrays
        ],
        axis=0,
    ).astype(
        np.float32,
        copy=False,
    )


    require(
        tensor.shape
        == (
            CACHE_CHANNEL_COUNT,
            size,
            size,
        ),
        (
            f'{row.region_id}: cached tensor shape '
            f'{tensor.shape} is invalid.'
        ),
    )


    require(
        np.isfinite(
            tensor
        ).all(),
        (
            f'{row.region_id}: cached tensor contains '
            'non-finite values.'
        ),
    )


    return tensor


# ============================================================
# SEPARATE UNCACHED REFERENCE IMPLEMENTATION
#
# This deliberately recomputes the arrays. It is used only on
# a small frozen sample to prove that caching did not alter
# tensor semantics.
# ============================================================

def uncached_crop_reference(
    array,
    centre_y,
    centre_x,
    size,
    pad_mode,
):

    half = (
        size
        // 2
    )


    pad = (
        half
        + 2
    )


    if pad_mode == 'constant':

        padded = np.pad(
            array,
            pad,
            mode='constant',
            constant_values=0,
        )

    else:

        padded = np.pad(
            array,
            pad,
            mode=pad_mode,
        )


    padded_y = (
        centre_y
        + pad
    )

    padded_x = (
        centre_x
        + pad
    )


    crop = padded[
        padded_y
        - half:
        padded_y
        + half
        + 1,

        padded_x
        - half:
        padded_x
        + half
        + 1,
    ]


    require(
        crop.shape
        == (
            size,
            size,
        ),
        (
            'Uncached reference crop has wrong shape: '
            f'{crop.shape}.'
        ),
    )


    return crop


def uncached_reference_channels(
    row_index,
    size,
    merged=False,
):

    row = fit.iloc[
        int(
            row_index
        )
    ]


    observed = read_observed_page(
        row[
            '_observed_path'
        ]
    )


    observable = build_page_observable_arrays(
        observed
    )


    page_frame = fit[
        fit[
            'page'
        ]
        .astype(str)
        .eq(
            str(
                row[
                    'page'
                ]
            )
        )
    ]


    if merged:

        mask = merged_region_mask(
            row,
            page_frame,
            observed.shape,
        )

    else:

        mask = single_region_mask(
            row,
            observed.shape,
        )


    distance = distance_channel(
        mask
    )


    centre_y, centre_x = region_centre(
        row
    )


    arrays_and_modes = [
        (
            observable[
                'observed'
            ],
            'reflect',
        ),

        (
            observable[
                'normalised'
            ],
            'reflect',
        ),

        (
            mask.astype(
                np.float32
            ),
            'constant',
        ),

        (
            distance.astype(
                np.float32
            ),
            'constant',
        ),

        (
            observable[
                'edge'
            ],
            'reflect',
        ),

        (
            observable[
                'valid_image'
            ],
            'constant',
        ),
    ]


    return np.stack(
        [
            uncached_crop_reference(
                array,
                centre_y,
                centre_x,
                size,
                pad_mode,
            )

            for array, pad_mode
            in arrays_and_modes
        ],
        axis=0,
    ).astype(
        np.float32
    )


# ============================================================
# FROZEN-SAMPLE CACHE EQUIVALENCE GATE
# ============================================================

equivalence_rng = np.random.default_rng(
    SEED
)


all_indices = np.arange(
    len(
        fit
    )
)


if len(
    all_indices
) <= CACHE_EQUIVALENCE_SAMPLE_SIZE:

    equivalence_indices = all_indices

else:

    equivalence_indices = np.sort(
        equivalence_rng.choice(
            all_indices,
            size=CACHE_EQUIVALENCE_SAMPLE_SIZE,
            replace=False,
        )
    )


equivalence_rows = []


for row_index in equivalence_indices:

    row = fit.iloc[
        int(
            row_index
        )
    ]


    for merged in [
        False,
        True,
    ]:

        for size in SCALES:

            reference = uncached_reference_channels(
                row_index,
                size,
                merged=merged,
            )


            cached = cached_channels(
                row_index,
                size,
                merged=merged,
            )


            equal = np.array_equal(
                reference,
                cached,
            )


            maximum_absolute_difference = float(
                np.max(
                    np.abs(
                        reference.astype(
                            np.float64
                        )
                        - cached.astype(
                            np.float64
                        )
                    )
                )
            )


            close = np.allclose(
                reference,
                cached,
                atol=CACHE_EQUIVALENCE_ATOL,
                rtol=CACHE_EQUIVALENCE_RTOL,
                equal_nan=True,
            )


            equivalence_rows.append({
                'row_index':
                    int(
                        row_index
                    ),

                'region_id':
                    row[
                        'region_id'
                    ],

                'page':
                    row[
                        'page'
                    ],

                'merged':
                    bool(
                        merged
                    ),

                'scale':
                    int(
                        size
                    ),

                'exact':
                    bool(
                        equal
                    ),

                'within_tolerance':
                    bool(
                        close
                    ),

                'max_absolute_difference':
                    maximum_absolute_difference,

                'reference_sha256':
                    array_sha256(
                        reference
                    ),

                'cached_sha256':
                    array_sha256(
                        cached
                    ),
            })


equivalence_frame = pd.DataFrame(
    equivalence_rows
)


equivalence_frame.to_csv(
    AUDIT
    / 'cached_tensor_equivalence.csv',
    index=False,
)


failed_equivalence = (
    equivalence_frame[
        ~equivalence_frame[
            'within_tolerance'
        ]
    ]
)


if len(
    failed_equivalence
):

    print(
        failed_equivalence.to_string(
            index=False
        )
    )


require(
    len(
        failed_equivalence
    )
    == 0,
    (
        'Cached tensor equivalence failed. '
        'Training has not started. See '
        'audit/cached_tensor_equivalence.csv.'
    ),
)


equivalence_manifest = {
    'seed':
        SEED,

    'sample_rows':
        int(
            len(
                equivalence_indices
            )
        ),

    'comparisons':
        int(
            len(
                equivalence_frame
            )
        ),

    'scales':
        [
            int(
                value
            )
            for value
            in SCALES
        ],

    'core_and_merged_tested':
        True,

    'exact_all':
        bool(
            equivalence_frame[
                'exact'
            ].all()
        ),

    'within_tolerance_all':
        bool(
            equivalence_frame[
                'within_tolerance'
            ].all()
        ),

    'atol':
        CACHE_EQUIVALENCE_ATOL,

    'rtol':
        CACHE_EQUIVALENCE_RTOL,

    'training_permitted':
        True,
}


with open(
    AUDIT
    / 'cached_tensor_equivalence_manifest.json',
    'w',
) as handle:

    json.dump(
        equivalence_manifest,
        handle,
        indent=2,
        sort_keys=True,
    )


print(
    '\nCACHED TENSOR EQUIVALENCE PASSED:',
    len(
        equivalence_frame
    ),
    'comparisons.',
)


# ============================================================
# DATASET AND LOADERS
# ============================================================

class CachedRegionDataset(
    Dataset
):

    def __init__(
        self,
        frame,
        indices,
        scales,
        merged=False,
    ):

        self.frame = frame

        self.indices = np.asarray(
            indices,
            dtype=np.int64,
        )

        self.scales = tuple(
            int(
                scale
            )
            for scale
            in scales
        )

        self.merged = bool(
            merged
        )


    def __len__(
        self,
    ):

        return len(
            self.indices
        )


    def __getitem__(
        self,
        position,
    ):

        row_index = int(
            self.indices[
                position
            ]
        )


        tensors = [
            torch.from_numpy(
                cached_channels(
                    row_index,
                    scale,
                    merged=self.merged,
                )
            )

            for scale
            in self.scales
        ]


        target = torch.tensor(
            int(
                self.frame.iloc[
                    row_index
                ][
                    '_class_id'
                ]
            ),
            dtype=torch.long,
        )


        return (
            tensors,
            target,
            row_index,
        )


def cached_collate(
    batch,
):

    number_of_scales = len(
        batch[
            0
        ][
            0
        ]
    )


    tensors = [
        torch.stack(
            [
                item[
                    0
                ][
                    scale_index
                ]

                for item
                in batch
            ],
            dim=0,
        )

        for scale_index
        in range(
            number_of_scales
        )
    ]


    targets = torch.stack(
        [
            item[
                1
            ]

            for item
            in batch
        ],
        dim=0,
    )


    row_indices = torch.tensor(
        [
            item[
                2
            ]

            for item
            in batch
        ],
        dtype=torch.long,
    )


    return (
        tensors,
        targets,
        row_indices,
    )


def cached_loader(
    indices,
    scales,
    merged,
    shuffle,
    seed,
):

    generator = torch.Generator()

    generator.manual_seed(
        seed
    )


    return DataLoader(
        CachedRegionDataset(
            fit,
            indices,
            scales,
            merged=merged,
        ),
        batch_size=BATCH,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        collate_fn=cached_collate,
        generator=generator,
        persistent_workers=False,
    )


# ============================================================
# MODEL
# ============================================================

class CachedBranch(
    nn.Module
):

    def __init__(
        self,
        number_of_channels=CACHE_CHANNEL_COUNT,
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Conv2d(
                number_of_channels,
                24,
                kernel_size=3,
                padding=1,
            ),

            nn.ReLU(),

            nn.BatchNorm2d(
                24
            ),

            nn.MaxPool2d(
                2
            ),

            nn.Conv2d(
                24,
                48,
                kernel_size=3,
                padding=1,
            ),

            nn.ReLU(),

            nn.BatchNorm2d(
                48
            ),

            nn.MaxPool2d(
                2
            ),

            nn.Conv2d(
                48,
                64,
                kernel_size=3,
                padding=1,
            ),

            nn.ReLU(),

            nn.AdaptiveAvgPool2d(
                1
            ),
        )


    def forward(
        self,
        tensor,
    ):

        return (
            self.network(
                tensor
            )
            .flatten(
                1
            )
        )


class CachedMultiScaleCNN(
    nn.Module
):

    def __init__(
        self,
        number_of_branches,
        number_of_classes,
    ):

        super().__init__()


        self.branches = nn.ModuleList(
            [
                CachedBranch()

                for _
                in range(
                    number_of_branches
                )
            ]
        )


        self.head = nn.Sequential(

            nn.Linear(
                64
                * number_of_branches,
                96,
            ),

            nn.ReLU(),

            nn.Dropout(
                0.15
            ),

            nn.Linear(
                96,
                number_of_classes,
            ),
        )


    def forward(
        self,
        tensors,
    ):

        embeddings = [
            branch(
                tensor
            )

            for branch, tensor
            in zip(
                self.branches,
                tensors,
            )
        ]


        return self.head(
            torch.cat(
                embeddings,
                dim=1,
            )
        )


# ============================================================
# SCALAR BASELINE
# ============================================================

X_scalar = (
    fit[
        scalar
    ]
    .apply(
        pd.to_numeric,
        errors='coerce',
    )
    .replace(
        [
            np.inf,
            -np.inf,
        ],
        np.nan,
    )
)


scalar_oof = np.full(
    len(
        fit
    ),
    -1,
    dtype=np.int64,
)


scalar_probs = np.full(
    (
        len(
            fit
        ),
        len(
            classes
        ),
    ),
    np.nan,
    dtype=np.float64,
)


scalar_fold_rows = []


for fold_number, (
    train_indices,
    test_indices,
) in enumerate(
    splits,
    start=1,
):

    print(
        f'\nSCALAR fold {fold_number}/{len(splits)}'
    )


    scalar_model = Pipeline([
        (
            'imputer',
            SimpleImputer(
                strategy='median'
            ),
        ),
        (
            'scaler',
            StandardScaler(),
        ),
        (
            'classifier',
            LogisticRegression(
                max_iter=5000,
                class_weight='balanced',
                random_state=SEED,
            ),
        ),
    ])


    scalar_model.fit(
        X_scalar.iloc[
            train_indices
        ],
        fit.iloc[
            train_indices
        ][
            '_class_id'
        ],
    )


    scalar_oof[
        test_indices
    ] = scalar_model.predict(
        X_scalar.iloc[
            test_indices
        ]
    )


    fold_probability = scalar_model.predict_proba(
        X_scalar.iloc[
            test_indices
        ]
    )


    # Align predict_proba columns if a fold presents classes in
    # a different estimator order.

    aligned_probability = np.zeros(
        (
            len(
                test_indices
            ),
            len(
                classes
            ),
        ),
        dtype=np.float64,
    )


    estimator_classes = (
        scalar_model.named_steps[
            'classifier'
        ]
        .classes_
    )


    for estimator_column, class_id in enumerate(
        estimator_classes
    ):

        aligned_probability[
            :,
            int(
                class_id
            ),
        ] = fold_probability[
            :,
            estimator_column
        ]


    scalar_probs[
        test_indices
    ] = aligned_probability


    scalar_path = (
        MODELS
        / f'scalar_fold{fold_number}.joblib'
    )


    joblib.dump(
        scalar_model,
        scalar_path,
    )


    scalar_fold_rows.append({
        'arm':
            'scalar',

        'fold':
            fold_number,

        'checkpoint':
            str(
                scalar_path
            ),

        'train_rows':
            int(
                len(
                    train_indices
                )
            ),

        'test_rows':
            int(
                len(
                    test_indices
                )
            ),
    })


require(
    (
        scalar_oof
        >= 0
    ).all(),
    (
        'Scalar OOF predictions are incomplete.'
    ),
)


require(
    np.isfinite(
        scalar_probs
    ).all(),
    (
        'Scalar OOF probabilities are incomplete or non-finite.'
    ),
)


# ============================================================
# CNN TRAINING
# ============================================================

def fit_predict_cached_cnn(
    scales,
    merged,
    arm,
):

    scales = tuple(
        int(
            scale
        )
        for scale
        in scales
    )


    oof_prediction = np.full(
        len(
            fit
        ),
        -1,
        dtype=np.int64,
    )


    oof_probability = np.full(
        (
            len(
                fit
            ),
            len(
                classes
            ),
        ),
        np.nan,
        dtype=np.float64,
    )


    history_rows = []
    model_rows = []


    for fold_number, (
        train_indices,
        test_indices,
    ) in enumerate(
        splits,
        start=1,
    ):

        fold_started = time.time()


        print(
            '\n'
            + '='
            * 80
        )

        print(
            f'{arm.upper()} '
            f'fold {fold_number}/{len(splits)}'
        )

        print(
            'Scales:',
            scales,
            '| merged:',
            merged,
        )

        print(
            'Train rows:',
            len(
                train_indices
            ),
            '| test rows:',
            len(
                test_indices
            ),
        )


        reset_seed(
            SEED
        )


        model = CachedMultiScaleCNN(
            number_of_branches=len(
                scales
            ),
            number_of_classes=len(
                classes
            ),
        ).to(
            DEVICE
        )


        train_targets = (
            fit.iloc[
                train_indices
            ][
                '_class_id'
            ]
            .to_numpy(
                dtype=np.int64
            )
        )


        class_counts = np.bincount(
            train_targets,
            minlength=len(
                classes
            ),
        )


        require(
            (
                class_counts
                > 0
            ).all(),
            (
                f'{arm} fold {fold_number}: one or more '
                'classes are absent from the training side: '
                f'{dict(zip(classes, class_counts.tolist()))}'
            ),
        )


        class_weights = torch.tensor(
            class_counts.sum()
            / np.maximum(
                class_counts,
                1,
            ),
            dtype=torch.float32,
            device=DEVICE,
        )


        loss_function = nn.CrossEntropyLoss(
            weight=class_weights
        )


        optimiser = torch.optim.AdamW(
            model.parameters(),
            lr=LR,
            weight_decay=WEIGHT_DECAY,
        )


        for epoch in range(
            1,
            EPOCHS
            + 1,
        ):

            epoch_started = time.time()

            model.train()

            total_loss = 0.0
            total_examples = 0


            training_loader = cached_loader(
                train_indices,
                scales,
                merged,
                shuffle=True,
                seed=(
                    SEED
                    + fold_number
                    * 1000
                    + epoch
                ),
            )


            for (
                input_tensors,
                targets,
                _,
            ) in training_loader:

                input_tensors = [
                    tensor.to(
                        DEVICE,
                        non_blocking=PIN_MEMORY,
                    )

                    for tensor
                    in input_tensors
                ]


                targets = targets.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )


                optimiser.zero_grad(
                    set_to_none=True
                )


                logits = model(
                    input_tensors
                )


                loss = loss_function(
                    logits,
                    targets,
                )


                loss.backward()

                optimiser.step()


                batch_size = int(
                    len(
                        targets
                    )
                )


                total_loss += (
                    float(
                        loss.detach().cpu()
                    )
                    * batch_size
                )


                total_examples += batch_size


            mean_loss = (
                total_loss
                / max(
                    total_examples,
                    1,
                )
            )


            epoch_seconds = (
                time.time()
                - epoch_started
            )


            history_rows.append({
                'arm':
                    arm,

                'fold':
                    fold_number,

                'epoch':
                    epoch,

                'mean_loss':
                    mean_loss,

                'seconds':
                    epoch_seconds,

                'train_rows':
                    int(
                        total_examples
                    ),
            })


            print(
                f'  epoch {epoch:02d}/{EPOCHS}: '
                f'loss={mean_loss:.6f} '
                f'rows={total_examples} '
                f'seconds={epoch_seconds:.2f}'
            )


        # ----------------------------------------------------
        # OOF prediction
        # ----------------------------------------------------

        model.eval()


        with torch.no_grad():

            testing_loader = cached_loader(
                test_indices,
                scales,
                merged,
                shuffle=False,
                seed=SEED,
            )


            for (
                input_tensors,
                _,
                row_indices,
            ) in testing_loader:

                input_tensors = [
                    tensor.to(
                        DEVICE,
                        non_blocking=PIN_MEMORY,
                    )

                    for tensor
                    in input_tensors
                ]


                probability = (
                    torch.softmax(
                        model(
                            input_tensors
                        ),
                        dim=1,
                    )
                    .cpu()
                    .numpy()
                    .astype(
                        np.float64
                    )
                )


                resolved_indices = (
                    row_indices.numpy()
                )


                oof_probability[
                    resolved_indices
                ] = probability


                oof_prediction[
                    resolved_indices
                ] = probability.argmax(
                    axis=1
                )


        # ----------------------------------------------------
        # Save immediately after the fold has completed
        # ----------------------------------------------------

        checkpoint_path = (
            MODELS
            / (
                f'{arm}_fold{fold_number}'
                f'_seed{SEED}.pt'
            )
        )


        checkpoint_payload = {
            'state_dict':
                model.state_dict(),

            'arm':
                arm,

            'fold':
                fold_number,

            'seed':
                SEED,

            'scales':
                list(
                    scales
                ),

            'merged':
                bool(
                    merged
                ),

            'channels':
                CACHE_CHANNELS,

            'classes':
                classes,

            'class_to_id':
                class_to_id,

            'cache_equivalence_manifest':
                equivalence_manifest,
        }


        torch.save(
            checkpoint_payload,
            checkpoint_path,
        )


        require(
            checkpoint_path.is_file(),
            (
                f'{arm} fold {fold_number}: checkpoint '
                f'was not written: {checkpoint_path}'
            ),
        )


        checkpoint_sha256 = hashlib.sha256(
            checkpoint_path.read_bytes()
        ).hexdigest()


        fold_seconds = (
            time.time()
            - fold_started
        )


        model_rows.append({
            'arm':
                arm,

            'fold':
                fold_number,

            'checkpoint':
                str(
                    checkpoint_path
                ),

            'checkpoint_sha256':
                checkpoint_sha256,

            'checkpoint_bytes':
                int(
                    checkpoint_path.stat().st_size
                ),

            'train_rows':
                int(
                    len(
                        train_indices
                    )
                ),

            'test_rows':
                int(
                    len(
                        test_indices
                    )
                ),

            'scales':
                '|'.join(
                    str(
                        scale
                    )
                    for scale
                    in scales
                ),

            'merged':
                bool(
                    merged
                ),

            'seconds':
                fold_seconds,
        })


        print(
            'Saved:',
            checkpoint_path,
        )

        print(
            'Checkpoint SHA-256:',
            checkpoint_sha256,
        )

        print(
            f'Fold seconds: {fold_seconds:.2f}'
        )


        del model
        del optimiser
        del loss_function


        gc.collect()


        if DEVICE == 'cuda':

            torch.cuda.empty_cache()


    require(
        (
            oof_prediction
            >= 0
        ).all(),
        (
            f'{arm}: OOF predictions are incomplete.'
        ),
    )


    require(
        np.isfinite(
            oof_probability
        ).all(),
        (
            f'{arm}: OOF probabilities contain '
            'missing/non-finite values.'
        ),
    )


    return (
        oof_prediction,
        oof_probability,
        pd.DataFrame(
            history_rows
        ),
        pd.DataFrame(
            model_rows
        ),
    )


# ============================================================
# RUN THE THREE CNN ARMS
# ============================================================

print(
    '\nStarting local33 OOF training.'
)


(
    local_oof,
    local_probs,
    local_history,
    local_models,
) = fit_predict_cached_cnn(
    scales=[
        33
    ],
    merged=False,
    arm='local33',
)


print(
    '\nStarting multiscale33_97_257 OOF training.'
)


(
    multi_oof,
    multi_probs,
    multi_history,
    multi_models,
) = fit_predict_cached_cnn(
    scales=SCALES,
    merged=False,
    arm='multiscale33_97_257',
)


print(
    '\nStarting multiscale_merged OOF training.'
)


(
    merged_oof,
    merged_probs,
    merged_history,
    merged_models,
) = fit_predict_cached_cnn(
    scales=SCALES,
    merged=True,
    arm='multiscale_merged',
)


# ============================================================
# WRITE TRAINING AND MODEL AUDITS
# ============================================================

training_history = pd.concat(
    [
        local_history,
        multi_history,
        merged_history,
    ],
    ignore_index=True,
)


training_history.to_csv(
    TABLES
    / 'training_history.csv',
    index=False,
)


model_inventory = pd.concat(
    [
        pd.DataFrame(
            scalar_fold_rows
        ),
        local_models,
        multi_models,
        merged_models,
    ],
    ignore_index=True,
)


model_inventory.to_csv(
    AUDIT
    / 'model_inventory.csv',
    index=False,
)


# ------------------------------------------------------------
# Immediate OOF smoke summary
# ------------------------------------------------------------

training_smoke_summary = []


for arm, prediction in {

    'scalar':
        scalar_oof,

    'local33':
        local_oof,

    'multiscale33_97_257':
        multi_oof,

    'multiscale_merged':
        merged_oof,

}.items():

    training_smoke_summary.append({

        'arm':
            arm,

        'balanced_accuracy':
            balanced_accuracy_score(
                fit[
                    '_class_id'
                ],
                prediction,
            ),

        'macro_f1':
            f1_score(
                fit[
                    '_class_id'
                ],
                prediction,
                average='macro',
            ),

        'rows':
            int(
                len(
                    fit
                )
            ),

        'pages':
            int(
                fit[
                    'page'
                ].nunique()
            ),
    })


training_smoke_summary = (
    pd.DataFrame(
        training_smoke_summary
    )
    .sort_values(
        'balanced_accuracy',
        ascending=False,
    )
)


training_smoke_summary.to_csv(
    TABLES
    / 'training_smoke_summary.csv',
    index=False,
)


print(
    '\nOOF training completed.'
)


print(
    training_smoke_summary.to_string(
        index=False
    )
)


print(
    '\nModel inventory:',
    AUDIT
    / 'model_inventory.csv',
)


print(
    'Training history:',
    TABLES
    / 'training_history.csv',
)


print(
    'Tensor equivalence:',
    AUDIT
    / 'cached_tensor_equivalence.csv',
)

Semantic classes: ['ANNOTATION', 'NAN', 'PAGE_SCAN_EDGE']
OOF folds: 5
Cached Page086: 3 regions, shape=(2882, 2167)
Cached Page088: 1 regions, shape=(2771, 2076)
Cached Page089: 1 regions, shape=(2511, 1886)
Cached Page090: 2 regions, shape=(2523, 1886)
Cached Page091: 2 regions, shape=(2567, 1936)
Cached Page092: 1 regions, shape=(2709, 2035)
Cached Page093: 3 regions, shape=(2664, 2010)
Cached Page094: 1 regions, shape=(2630, 1969)
Cached Page095: 2 regions, shape=(2546, 1903)
Cached Page097: 3 regions, shape=(2512, 1870)
Cached Page100: 1 regions, shape=(2677, 1985)
Cached Page102: 1 regions, shape=(2607, 1936)
Cached Page103: 1 regions, shape=(2523, 1886)
Cached Page104: 2 regions, shape=(2523, 1886)
Cached Page105: 2 regions, shape=(2550, 1903)
Cached Page106: 2 regions, shape=(2555, 1903)
Cached Page107: 2 regions, shape=(2467, 1836)
Cached Page108: 1 regions, shape=(2500, 1861)
Cached Page109: 1 regions, shape=(2664, 1977)
Cached Page110: 1 regions, shape=(2772, 2060)


In [ ]:
# @title
y=fit._class_id.to_numpy();arms={'scalar':(scalar_oof,scalar_probs),'local33':(local_oof,local_probs),'multiscale33_97_257':(multi_oof,multi_probs),'multiscale_merged':(merged_oof,merged_probs)}
summary=[];metrics=[];oof=fit[['region_id','page','semantic_label','annotation_family','original_semantic_label']].copy()
for arm,(pred,prob) in arms.items():
    summary.append({'arm':arm,'balanced_accuracy':balanced_accuracy_score(y,pred),'macro_f1':f1_score(y,pred,average='macro'),'weighted_f1':f1_score(y,pred,average='weighted'),'rows':len(y),'pages':fit.page.nunique()})
    report=classification_report(y,pred,labels=list(range(len(classes))),target_names=classes,output_dict=True,zero_division=0)
    for c in classes:metrics.append({'arm':arm,'class':c,**{k:report[c][k] for k in ['precision','recall','f1-score','support']}})
    oof[f'pred_{arm}']=[classes[i] for i in pred]
    oof[f'confidence_{arm}']=prob.max(1)
    for i,c in enumerate(classes):oof[f'p_{arm}_{c}']=prob[:,i]
    cm=confusion_matrix(y,pred,labels=range(len(classes)))
    plt.figure(figsize=(8,7));plt.imshow(cm,cmap='Blues');plt.xticks(range(len(classes)),classes,rotation=45,ha='right');plt.yticks(range(len(classes)),classes);plt.xlabel('Predicted');plt.ylabel('Human/adjudicated');plt.title(f'59R5 OOF confusion: {arm}')
    for i in range(len(classes)):
        for j in range(len(classes)):plt.text(j,i,str(cm[i,j]),ha='center',va='center')
    plt.tight_layout();plt.savefig(VISUAL/f'confusion_{arm}.png',dpi=170);plt.close()
summary=pd.DataFrame(summary).sort_values('balanced_accuracy',ascending=False);summary.to_csv(TABLES/'model_summary.csv',index=False);pd.DataFrame(metrics).to_csv(TABLES/'class_metrics.csv',index=False);oof.to_csv(TABLES/'out_of_fold_predictions.csv',index=False)
print(summary.to_string(index=False))


In [ ]:
# @title
# Calibrated abstention analysis. No operational threshold is authorised here.
best=summary.iloc[0].arm;pred,prob=arms[best]
rows=[]
for t in np.linspace(.50,.99,50):
    mx=prob.max(1);decision=np.where(mx>=t,pred,-1);selected=decision>=0
    rows.append({'arm':best,'confidence_threshold':float(t),'selected':int(selected.sum()),'coverage':float(selected.mean()),'balanced_accuracy_selected':balanced_accuracy_score(y[selected],decision[selected]) if selected.sum() and len(np.unique(y[selected]))>1 else np.nan,'macro_f1_selected':f1_score(y[selected],decision[selected],average='macro') if selected.sum() else np.nan,'errors':int(np.sum(decision[selected]!=y[selected]))})
abst=pd.DataFrame(rows);abst.to_csv(TABLES/'abstention_frontier.csv',index=False)
plt.figure(figsize=(8,6));plt.plot(abst.coverage,abst.balanced_accuracy_selected,marker='o');plt.xlabel('Coverage');plt.ylabel('Balanced accuracy among selected');plt.title(f'59R5 development abstention frontier: {best}');plt.tight_layout();plt.savefig(VISUAL/'abstention_frontier.png',dpi=170);plt.close()

# Family diagnostic head: fit only annotation/mixed rows and sufficiently supported families.
af=fit[fit.semantic_label.isin(['ANNOTATION','MIXED']) & ~fit.annotation_family.isin(['','NOT_APPLICABLE'])].copy()
af_support=af.groupby('annotation_family',as_index=False).agg(regions=('region_id','size'),pages=('page','nunique'));af_support['eligible']=(af_support.regions>=8)&(af_support.pages>=4);af_support.to_csv(TABLES/'annotation_family_support.csv',index=False)
# Keep family evaluation descriptive if support is too small; semantic ownership remains primary.
print('Best semantic arm:',best)
print('Annotation-family support:');print(af_support.to_string(index=False))


In [ ]:
# @title
# Diagnostic contact sheets for wins, regressions, uncertain/page-edge cases.
def contact(rows,name,n=20):
    rows=rows.head(n)
    if len(rows)==0:return
    cols=4;rr=int(np.ceil(len(rows)/cols));fig,axes=plt.subplots(rr,cols,figsize=(16,5*rr),squeeze=False)
    for ax in axes.ravel():ax.axis('off')
    for ax,(_,r) in zip(axes.ravel(),rows.iterrows()):
        im=load_page(r._observed_path);cy,cx=centre(r);size=257;crop=crop_fixed(im,cy,cx,size);ax.imshow(crop,cmap='gray',vmin=0,vmax=255);ax.set_title(f"{r.page} | human={r.semantic_label} | scalar={r.pred_scalar} | multi={r.pred_multi} | merged={r.pred_merged}",fontsize=8);ax.axis('off')
    plt.tight_layout();plt.savefig(VISUAL/f'{name}.png',dpi=150,bbox_inches='tight');plt.close()
view=fit.copy();view['pred_scalar']=[classes[i] for i in scalar_oof];view['pred_multi']=[classes[i] for i in multi_oof];view['pred_merged']=[classes[i] for i in merged_oof]
contact(view[(view.pred_scalar!=view.semantic_label)&(view.pred_multi==view.semantic_label)],'multiscale_wins')
contact(view[(view.pred_scalar==view.semantic_label)&(view.pred_multi!=view.semantic_label)],'multiscale_regressions')
contact(view[view.semantic_label.isin(['UNCERTAIN','PAGE_SCAN_EDGE'])],'uncertain_and_page_edges')
contact(view[(view.pred_multi!=view.semantic_label)&(view.pred_merged==view.semantic_label)],'region_merge_wins')
contact(view[(view.pred_multi==view.semantic_label)&(view.pred_merged!=view.semantic_label)],'region_merge_regressions')


In [ ]:
# @title
# Final development-only verdict and upload package.
bestrow=summary.iloc[0]
verdict={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'59R5','development_only':True,'promotion_authority':False,'source_59R4_run':str(R4),'source_59R4A_run':str(R4A) if R4A else None,'adjudication_used':ADJ_PATH is not None,'rows_total_evidence':len(completed),'rows_supervised_fit':len(fit),'pages':int(fit.page.nunique()),'classes_fitted':classes,'classes_audit_only':support.loc[~support.eligible_for_supervised_fit,'semantic_label'].tolist(),'best_arm':str(bestrow.arm),'best_balanced_accuracy':float(bestrow.balanced_accuracy),'best_macro_f1':float(bestrow.macro_f1),'region_fragment_excluded':int((completed.fit_exclusion_reason=='REGION_FRAGMENT').sum()),'minimum_class_regions':MIN_CLASS_REGIONS,'minimum_class_pages':MIN_CLASS_PAGES,'raw_coordinates_excluded':True,'page_identity_excluded':True,'source_group_excluded':True,'clean_truth_used_as_input':False}
atomic_json(OUT/'notebook59R5_verdict.json',verdict)
(OUT/'executive_summary.txt').write_text(chr(10).join(['NOTEBOOK 59R5 EXPANDED MULTI-SCALE REGION CHALLENGER','',f'Rows fitted: {len(fit)} across {fit.page.nunique()} pages',f'Classes fitted: {classes}',f'Best arm: {bestrow.arm}',f'Balanced accuracy: {bestrow.balanced_accuracy:.6f}',f'Macro F1: {bestrow.macro_f1:.6f}','','Development evidence only. No correction policy, threshold or production promotion is authorised.']))
env={'python':sys.version,'torch':torch.__version__,'cuda':torch.version.cuda,'device':DEVICE,'opencv':cv2.__version__,'numpy':np.__version__,'pandas':pd.__version__,'platform':platform.platform()};atomic_json(OUT/'software_environment.json',env)

artefacts=[OUT/'executive_summary.txt',OUT/'notebook59R5_verdict.json',TABLES/'model_summary.csv',TABLES/'class_metrics.csv',TABLES/'semantic_class_support.csv',TABLES/'out_of_fold_predictions.csv',TABLES/'abstention_frontier.csv',AUDIT/'predictor_firewall.json']
artefacts += sorted(VISUAL.glob('*.png'))
manifest=[]
for p in artefacts:
    require(p.exists(),f'Missing upload artefact {p}');dst=UPLOAD/p.name;shutil.copy2(p,dst);manifest.append({'file':p.name,'sha256':sha256_file(dst),'bytes':dst.stat().st_size})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
zip_path=OUT/'notebook59R5_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.iterdir():z.write(p,p.name)
print('59R5 complete:',OUT);print('Upload bundle:',zip_path)
